# Cultural Adaptation Pipeline
**Task:** Adapt source-culture text (Western/American) → Indian subregion cultures  
**Dataset:** EECC (shaily99/eecc)  
**LLM:** Llama 3.3 70B via Groq API

In [2]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install groq datasets pandas tqdm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 1.0 MB/s eta 0:00:00


In [ ]:
# ── Cell 2: Set your Groq API key ─────────────────────────────────────────────
import os
os.environ["GROQ_API_KEY"] = "" # <-- paste your key here
# Get a free key at: https://console.groq.com

In [4]:
# ── Cell 3: Imports ───────────────────────────────────────────────────────────
import time
import json
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset
from groq import Groq

In [5]:
import warnings
warnings.filterwarnings('ignore', message="`trust_remote_code` is not supported anymore.")

# ── Cell 4: Load EECC dataset ─────────────────────────────────────────────────
print("Loading EECC dataset...")
dataset = load_dataset("shaily99/eecc", 'Story responses') # Changed to 'Story responses' for nationality and story data
print(dataset)
print("\nColumn names:", dataset[list(dataset.keys())[0]].column_names)
print("\nSample row:")
print(dataset[list(dataset.keys())[0]][0])

Loading EECC dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

stories/responses/story_responses_1000_t(…):   0%|          | 0.00/67.4M [00:00<?, ?B/s]

stories/responses/story_responses_1000_t(…):   0%|          | 0.00/68.4M [00:00<?, ?B/s]

stories/responses/story_responses_1000_t(…):   0%|          | 0.00/72.0M [00:00<?, ?B/s]

... (6 outputs left out)

stories/responses/story_responses_1000_t(…):   0%|          | 0.00/152M [00:00<?, ?B/s]

stories/responses/story_responses_1000_t(…):   0%|          | 0.00/90.7M [00:00<?, ?B/s]

Generating stories split:   0%|          | 0/74294 [00:00<?, ? examples/s]

DatasetDict({
    stories: Dataset({
        features: ['max_tokens', 'temperature', 'prompt_template', 'topic_category', 'concept', 'type_of_identity', 'identity', 'prompt', 'response_1', 'response_2', 'response_3', 'response_4', 'response_5'],
        num_rows: 74294
    })
})

Column names: ['max_tokens', 'temperature', 'prompt_template', 'topic_category', 'concept', 'type_of_identity', 'identity', 'prompt', 'response_1', 'response_2', 'response_3', 'response_4', 'response_5']

Sample row:
{'max_tokens': 1000, 'temperature': 0.3, 'prompt_template': "Write a children's story about {concept} for {vowel} {identity} kid in English.", 'topic_category': 'stories', 'concept': 'honesty', 'type_of_identity': 'country', 'identity': 'Albanian', 'prompt': "Write a children's story about honesty for an Albanian kid in English.", 'response_1': 'In the quaint village of Skanderbeg, nestled amidst rolling hills and sparkling rivers, lived a young Albanian boy named Ismail. Ismail was known througho

In [6]:
# ── Cell 5: Explore & pick source texts ───────────────────────────────────────
# EECC contains stories generated for various nationalities.
# We'll use American/Western stories as source culture.

split = list(dataset.keys())[0]  # usually 'train'
df = dataset[split].to_pandas()

print("Shape:", df.shape)
print("\nUnique nationalities sample:", df['nationality'].unique()[:20] if 'nationality' in df.columns else "column not found")
print("\nColumns:", df.columns.tolist())
df.head(3)

Shape: (74294, 13)

Unique nationalities sample: column not found

Columns: ['max_tokens', 'temperature', 'prompt_template', 'topic_category', 'concept', 'type_of_identity', 'identity', 'prompt', 'response_1', 'response_2', 'response_3', 'response_4', 'response_5']


,max_tokens,temperature,prompt_template,topic_category,concept,type_of_identity,identity,prompt,response_1,response_2,response_3,response_4,response_5
0,1000,0.3,Write a children's story about {concept} for {...,stories,honesty,country,Albanian,Write a children's story about honesty for an ...,"In the quaint village of Skanderbeg, nestled a...","In the heart of the Albanian village, lived a ...","In the quaint village of Kalabash, nestled ami...","In the quaint village of Kalabash, lived a lit...","In the heart of the Albanian village, lived a ..."
1,1000,0.3,Write a children's story about {concept} for {...,stories,honesty,country,Algerian,Write a children's story about honesty for an ...,"Once upon a time, in a vibrant village nestled...","In the heart of the bustling city, lived a lit...","Once upon a time, in a land filled with colorf...","In the heart of the bustling city, lived a lit...","In the heart of the Sahara, lived a little boy..."
2,1000,0.3,Write a children's story about {concept} for {...,stories,honesty,country,American,Write a children's story about honesty for an ...,"In the quaint town of Willow Creek, lived a yo...","In the quaint town of Willow Creek, there live...","In the quaint town of Willow Creek, lived a cu...","In the quaint town of Willow Creek, lived a yo...","In the quaint town of Willow Creek, lived a yo..."


In [7]:
# ── Cell 6: Reshape dataset and filter Western source texts ───────────────────
import pandas as pd

# The 'Story responses' split is already in a long format, so melting is not needed.
# Directly load the split as df_long.
df_long = dataset[list(dataset.keys())[0]].to_pandas()

print(f"Columns in df_long after loading: {df_long.columns.tolist()}")

# Rename 'response_1' to 'story' and 'identity' to 'nationality' to match subsequent code
if 'response_1' in df_long.columns:
    df_long = df_long.rename(columns={'response_1': 'story'})
if 'identity' in df_long.columns:
    df_long = df_long.rename(columns={'identity': 'nationality'})

# Ensure 'story' column is treated as string for strip operation
df_long = df_long.dropna(subset=['story'])
df_long = df_long[df_long['story'].astype(str).str.strip() != '']

print(f"Long format shape: {df_long.shape}")
print(f"Nationalities sample: {df_long['nationality'].unique()[:10]}")
# The 'Story responses' config has 'topic_category'
print(f"Topics sample: {df_long['topic_category'].unique()[:5]}")

# Filter Western source texts
TEXT_COL   = "story"
NATION_COL = "nationality" # This column is now directly available from the dataset

SOURCE_CULTURES = ["American", "British", "Australian"]

source_df = df_long[df_long[NATION_COL].isin(SOURCE_CULTURES)].copy()
print(f"\nSource texts found: {len(source_df)}")

N_SAMPLES = 30
# Sample only if there are enough source texts available
if not source_df.empty and len(source_df) >= N_SAMPLES:
    source_sample = source_df.sample(n=N_SAMPLES, random_state=42).reset_index(drop=True)
    print(f"Sampled {N_SAMPLES} stories")
    print(f"\nExample:\n{source_sample[TEXT_COL][0][:400]}")
elif not source_df.empty:
    print(f"Not enough source texts ({len(source_df)}) to sample {N_SAMPLES} stories. Sampling all available texts.")
    source_sample = source_df.reset_index(drop=True)
    print(f"Sampled {len(source_sample)} stories")
    print(f"\nExample:\n{source_sample[TEXT_COL][0][:400]}")
else:
    source_sample = pd.DataFrame(columns=source_df.columns) # Create an empty DataFrame with the same columns
    print("No source texts available after filtering.")


Columns in df_long after loading: ['max_tokens', 'temperature', 'prompt_template', 'topic_category', 'concept', 'type_of_identity', 'identity', 'prompt', 'response_1', 'response_2', 'response_3', 'response_4', 'response_5']
Long format shape: (74294, 13)
Nationalities sample: ['Albanian' 'Algerian' 'American' 'Andorran' 'Angolan' 'Antiguan'
 'Argentine' 'Armenian' 'Australian' 'Austrian']
Topics sample: ['stories']

Source texts found: 1155
Sampled 30 stories

Example:
 (approx. 250-300 words)\n \n Title: The Little Koala's Big Mistake\n \n Once upon a time, in the land of Australia, there lived a little koala named Kip. Kip loved to play with his friends and climb the tall eucalyptus trees.\n \n One day, Kip told his friends that he had climbed the tallest tree in the forest. His friends were very impressed and asked Kip to show them the tree.\n \n But, Kip had


In [8]:
# ── Cell 7: Define target cultures (Indian subregions) ────────────────────────
# You can extend this list. Covers linguistic/geographic diversity.
TARGET_CULTURES = [
    {"region": "Tamil Nadu",    "language": "Tamil",   "notes": "Dravidian culture, Carnatic music, rice-based cuisine, Pongal festival"},
    {"region": "Punjab",        "language": "Punjabi", "notes": "North Indian, Bhangra, wheat-based cuisine, Lohri festival, agricultural"},
    {"region": "West Bengal",   "language": "Bengali", "notes": "Eastern India, Durga Puja, fish-rice cuisine, literary tradition"},
    {"region": "Maharashtra",   "language": "Marathi", "notes": "Western India, Ganesh Chaturthi, Vada pav, cosmopolitan Mumbai"},
    {"region": "Rajasthan",     "language": "Hindi",   "notes": "Desert culture, folk music, Teej festival, spiced cuisine"},
    {"region": "Kerala",        "language": "Malayalam","notes": "Coastal, Onam festival, seafood, matrilineal traditions, high literacy"},
]

print(f"Target cultures defined: {len(TARGET_CULTURES)}")
for c in TARGET_CULTURES:
    print(f"  - {c['region']} ({c['language']})")

Target cultures defined: 6
  - Tamil Nadu (Tamil)
  - Punjab (Punjabi)
  - West Bengal (Bengali)
  - Maharashtra (Marathi)
  - Rajasthan (Hindi)
  - Kerala (Malayalam)


In [9]:
# ── Cell 8: Adaptation prompt ─────────────────────────────────────────────────
def build_prompt(source_text: str, target: dict) -> str:
    return f"""You are an expert in Indian regional cultures. Your task is to culturally adapt a story.

## Source story (Western/American culture)
{source_text}

## Target culture
Region: {target['region']}, India
Language spoken: {target['language']}
Cultural notes: {target['notes']}

## Instructions
Rewrite the story so it feels natural and authentic to someone from {target['region']}. You must:
1. Replace cultural specific items (foods, festivals, places, names, occupations, sports) with {target['region']}-appropriate equivalents
2. Adapt social norms, family dynamics, and values to match the culture — not just swap words
3. Preserve the core plot, intent, and emotional tone of the original
4. Write in fluent English (do NOT translate to {target['language']})
5. Do NOT add explanatory notes or commentary — output only the adapted story

## Adapted story"""

# Test prompt
print(build_prompt(source_sample[TEXT_COL][0][:300] + "...", TARGET_CULTURES[0]))

You are an expert in Indian regional cultures. Your task is to culturally adapt a story.

## Source story (Western/American culture)
 (approx. 250-300 words)\n \n Title: The Little Koala's Big Mistake\n \n Once upon a time, in the land of Australia, there lived a little koala named Kip. Kip loved to play with his friends and climb the tall eucalyptus trees.\n \n One day, Kip told his friends that he had climbed the tallest tree i...

## Target culture
Region: Tamil Nadu, India
Language spoken: Tamil
Cultural notes: Dravidian culture, Carnatic music, rice-based cuisine, Pongal festival

## Instructions
Rewrite the story so it feels natural and authentic to someone from Tamil Nadu. You must:
1. Replace cultural specific items (foods, festivals, places, names, occupations, sports) with Tamil Nadu-appropriate equivalents
2. Adapt social norms, family dynamics, and values to match the culture — not just swap words
3. Preserve the core plot, intent, and emotional tone of the original
4. Writ

In [10]:
# ── Cell 9: Groq API call ─────────────────────────────────────────────────────
client = Groq(api_key=os.environ["GROQ_API_KEY"])

def adapt_text(source_text: str, target: dict, max_retries: int = 3) -> str:
    prompt = build_prompt(source_text, target)
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.7,
                max_tokens=1024,
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            print(f"  Attempt {attempt+1} failed: {e}")
            time.sleep(2 ** attempt)  # exponential backoff
    return "[ADAPTATION_FAILED]"

# Quick test
test_out = adapt_text(source_sample[TEXT_COL][0], TARGET_CULTURES[0])
print("=== ORIGINAL ===")
print(source_sample[TEXT_COL][0][:400])
print("\n=== ADAPTED (Tamil Nadu) ===")
print(test_out[:400])

=== ORIGINAL ===
 (approx. 250-300 words)\n \n Title: The Little Koala's Big Mistake\n \n Once upon a time, in the land of Australia, there lived a little koala named Kip. Kip loved to play with his friends and climb the tall eucalyptus trees.\n \n One day, Kip told his friends that he had climbed the tallest tree in the forest. His friends were very impressed and asked Kip to show them the tree.\n \n But, Kip had

=== ADAPTED (Tamil Nadu) ===
The Little Kuruvai's Big Mistake

Once upon a time, in a small village in Tamil Nadu, there lived a little boy named Kannan. Kannan loved to play with his friends and climb the tall palm trees that swayed gently in the breeze. During the Pongal festival, Kannan told his friends that he had climbed the tallest palm tree in the village and had seen the entire village from the top. His friends were v


In [ ]:
# ── Cell 10: Run full adaptation loop ─────────────────────────────────────────
# WARNING: This runs N_SAMPLES × len(TARGET_CULTURES) API calls
# = 30 × 6 = 180 calls. Should stay well within Groq free tier limits.
# Groq free tier: ~30 req/min, ~14,400 req/day for llama-3.3-70b

DELAY_BETWEEN_CALLS = 2.5  # seconds — adjust if you hit rate limits

results = []

for i, row in tqdm(source_sample.iterrows(), total=len(source_sample), desc="Stories"):
    source_text = row[TEXT_COL]
    source_culture = row[NATION_COL]

    for target in TARGET_CULTURES:
        adapted = adapt_text(source_text, target)
        results.append({
            "story_id":       i,
            "source_culture": source_culture,
            "source_text":    source_text,
            "target_region":  target["region"],
            "target_language":target["language"],
            "adapted_text":   adapted,
        })
        time.sleep(DELAY_BETWEEN_CALLS)

print(f"\nDone! Total adaptations: {len(results)}")

In [ ]:
# Load previously saved results 
import pandas as pd
from google.colab import files

uploaded = files.upload()  # upload adapted_texts_partial.csv

existing_df = pd.read_csv("adapted_texts_partial.csv")
results = existing_df.to_dict("records")
print(f"Loaded {len(results)} existing results")

Saving adapted_texts_partial.csv to adapted_texts_partial.csv
Loaded 130 existing results


In [ ]:
import time
from tqdm import tqdm

ALREADY_DONE = set(range(20))  # stories 0-19 already done

for i, row in tqdm(source_sample.iterrows(), total=len(source_sample), desc="Stories"):
    if i in ALREADY_DONE:
        continue
    source_text = row[TEXT_COL]
    source_culture = row[NATION_COL]
    for target in TARGET_CULTURES:
        adapted = adapt_text(source_text, target)
        results.append({
            "story_id":        i,
            "source_culture":  source_culture,
            "source_text":     source_text,
            "target_region":   target["region"],
            "target_language": target["language"],
            "adapted_text":    adapted,
        })
        time.sleep(2.5)

print(f"Total results now: {len(results)}")

In [ ]:
# ── Cell 11: Save results ─────────────────────────────────────────────────────
results_df = pd.DataFrame(results)
results_df.to_csv("adapted_texts.csv", index=False)
print(f"Saved {len(results_df)} rows to adapted_texts.csv")
print(results_df.groupby('target_region').size())
results_df.head(3)

Saved 130 rows to adapted_texts.csv
target_region
Kerala         21
Maharashtra    22
Punjab         22
Rajasthan      21
Tamil Nadu     22
West Bengal    22
dtype: int64


,story_id,source_culture,source_text,target_region,target_language,adapted_text
0,0,Australian,(approx. 250-300 words)\n \n Title: The Littl...,Tamil Nadu,Tamil,The Little Kuravan's Big Mistake\n\nOnce upon ...
1,0,Australian,(approx. 250-300 words)\n \n Title: The Littl...,Punjab,Punjabi,The Little Farmer's Big Mistake\n\nOnce upon a...
2,0,Australian,(approx. 250-300 words)\n \n Title: The Littl...,West Bengal,Bengali,The Little Monk's Big Mistake\n\nOnce upon a t...


In [15]:
#cell 11-new
# Save combined results
results_df = pd.DataFrame(results)
print(f"Total rows: {len(results_df)}")
print(f"Failed: {(results_df['adapted_text'] == '[ADAPTATION_FAILED]').sum()}")
print(f"Successful: {(results_df['adapted_text'] != '[ADAPTATION_FAILED]').sum()}")
results_df.to_csv("adapted_texts_final.csv", index=False)
print("Saved!")

Total rows: 190
Failed: 71
Successful: 119
Saved!


In [16]:
failed_df = results_df[results_df["adapted_text"] == "[ADAPTATION_FAILED]"]
print(f"Failed story IDs: {sorted(failed_df['story_id'].unique())}")
print(f"\nFailed counts per story:")
print(failed_df.groupby("story_id").size())
print(f"\nFailed counts per region:")
print(failed_df.groupby("target_region").size())

Failed story IDs: [np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(27), np.int64(28), np.int64(29)]

Failed counts per story:
story_id
12    2
13    6
14    6
15    6
16    6
17    6
18    6
19    6
20    6
21    4
27    5
28    6
29    6
dtype: int64

Failed counts per region:
target_region
Kerala         12
Maharashtra    12
Punjab         12
Rajasthan      12
Tamil Nadu     11
West Bengal    12
dtype: int64


In [ ]:
# ── Fix failures with new API key ─────────────────────────────────────────────
import time
from tqdm import tqdm
from groq import Groq

# Paste your NEW Groq API key here
NEW_API_KEY = "" 
new_client = Groq(api_key=NEW_API_KEY)

def adapt_text_new(source_text, target, max_retries=3):
    prompt = build_prompt(source_text, target)
    for attempt in range(max_retries):
        try:
            response = new_client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.7,
                max_tokens=1024,
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            print(f"  Attempt {attempt+1} failed: {e}")
            time.sleep(2 ** attempt)
    return "[ADAPTATION_FAILED]"

# Build lookup of what failed
failed_df = results_df[results_df["adapted_text"] == "[ADAPTATION_FAILED]"].copy()
failed_lookup = {}
for _, row in failed_df.iterrows():
    sid = int(row["story_id"])
    if sid not in failed_lookup:
        failed_lookup[sid] = []
    failed_lookup[sid].append(row["target_region"])

print(f"Need to redo {len(failed_df)} combinations across stories: {sorted(failed_lookup.keys())}")

# Re-run only failed combinations
new_results = []
for story_id, regions in tqdm(failed_lookup.items(), desc="Stories"):
    row = source_sample.loc[story_id]
    for target in TARGET_CULTURES:
        if target["region"] not in regions:
            continue
        adapted = adapt_text_new(row[TEXT_COL], target)
        new_results.append({
            "story_id":        story_id,
            "source_culture":  row[NATION_COL],
            "source_text":     row[TEXT_COL],
            "target_region":   target["region"],
            "target_language": target["language"],
            "adapted_text":    adapted,
        })
        time.sleep(3)  # slightly more conservative

print(f"\nNew results generated: {len(new_results)}")
failed_again = sum(1 for r in new_results if r["adapted_text"] == "[ADAPTATION_FAILED]")
print(f"Still failed: {failed_again}")

In [ ]:
# ── Cell 12: Download the CSV from Colab ──────────────────────────────────────
from google.colab import files
files.download("adapted_texts.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Next steps
Once you have `adapted_texts.csv`, the next notebook will build the **evaluation pipeline**:
- CSI replacement rate (how many cultural items were actually swapped)
- Semantic similarity (BERTScore — does it preserve meaning?)
- LLM-as-judge scoring (cultural authenticity + faithfulness)
- Aggregate results by region